In [ ]:
from dataclasses import dataclass
import math

import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt


@dataclass(frozen=True)
class TestCurve:
    key: str
    name: str
    curve_type: str
    x: object
    values: object


@dataclass(frozen=True)
class TestParameter:
    name: str
    plot_name: str
    curves: tuple

    @property
    def curve_types(self):
        return tuple(curve.curve_type for curve in self.curves)

    def get(self, curve_type):
        for curve in self.curves:
            if curve.curve_type == curve_type:
                return curve
        raise KeyError(f"Unknown curve type {curve_type!r}. Available: {self.curve_types}")

    def values(self, curve_type):
        return self.get(curve_type).values

    def __getitem__(self, key):
        if isinstance(key, int):
            return self.curves[key]
        return self.get(key)

    def __iter__(self):
        return iter(self.curves)

    def __len__(self):
        return len(self.curves)


@dataclass(frozen=True)
class TestDataset:
    parameters: tuple

    @property
    def names(self):
        return tuple(parameter.name for parameter in self.parameters)

    @property
    def all_curves(self):
        return tuple(curve for parameter in self.parameters for curve in parameter.curves)

    def get(self, name):
        for parameter in self.parameters:
            if parameter.name == name:
                return parameter
        raise KeyError(f"Unknown parameter {name!r}. Available: {self.names}")

    def __getitem__(self, key):
        if isinstance(key, int):
            return self.parameters[key]
        return self.get(key)

    def __iter__(self):
        return iter(self.parameters)

    def __len__(self):
        return len(self.parameters)


@dataclass(frozen=True)
class TestRunResult:
    dataset: object
    results: tuple
    fig: object
    axes: object


@dataclass(frozen=True)
class TestVariableSpec:
    key: str
    name: str
    curve_types: object = None
    complex_values: bool = False
    positive: bool = True
    zeros: bool = True


NONZERO_EPS = 1.0e-6


def _replace_zeros(values, replacement=NONZERO_EPS):
    values = jnp.asarray(values)
    replacement = jnp.asarray(replacement, dtype=values.dtype)
    return jnp.where(values == 0, replacement, values)


def _prepare_curve_values(values, positive=True, zeros=True):
    if positive:
        values = jnp.abs(values)
    if not zeros:
        values = _replace_zeros(values)
    return values


def _curve_templates(length, positive=True, zeros=True):
    x = jnp.linspace(0.0, 1.0, length, dtype=jnp.float32)
    curve_x = x if zeros else _replace_zeros(x)
    center = 0.5
    width = 0.12
    delta = jnp.zeros(length, dtype=jnp.float32).at[length // 2].set(1.0)

    curves = {
        "constant": jnp.ones_like(curve_x),
        "zero": jnp.zeros_like(curve_x),
        "identity function": curve_x,
        "linear": 2.0 * curve_x - 1.0,
        "quadratic": curve_x**2,
        "power law": jnp.sqrt(curve_x),
        "gaussian": jnp.exp(-0.5 * ((curve_x - center) / width) ** 2),
        "delta": delta,
        "square pulse": jnp.where((curve_x >= 0.35) & (curve_x <= 0.65), 1.0, 0.0),
        "smooth step": 0.5 * (1.0 + jnp.tanh((curve_x - center) / 0.08)),
        "sinusoidal": 0.5 * (1.0 - jnp.cos(2.0 * jnp.pi * curve_x)),
        "lorentzian": 1.0 / (1.0 + ((curve_x - center) / width) ** 2),
        "exponential decay": jnp.exp(-4.0 * curve_x),
    }
    curves = {name: _prepare_curve_values(values, positive=positive, zeros=zeros) for name, values in curves.items()}
    return x, curves


def _with_complex_values(curves, x, positive=True, zeros=True):
    center = 0.5
    width = 0.12
    shifted_center = 0.62
    delta_imag = jnp.zeros(x.shape, dtype=jnp.float32).at[x.shape[0] // 3].set(1.0)
    imag_curves = {
        "constant": 0.5 * jnp.ones_like(x),
        "zero": jnp.zeros_like(x),
        "identity function": 1.0 - x,
        "linear": 0.75 - 1.5 * x,
        "quadratic": (1.0 - x)**2,
        "power law": jnp.sqrt(1.0 - x),
        "gaussian": jnp.exp(-0.5 * ((x - shifted_center) / width) ** 2),
        "delta": delta_imag,
        "square pulse": jnp.where((x >= 0.55) & (x <= 0.85), 1.0, 0.0),
        "smooth step": 0.5 * (1.0 - jnp.tanh((x - center) / 0.08)),
        "sinusoidal": 0.5 * (1.0 + jnp.sin(2.0 * jnp.pi * x)),
        "lorentzian": 1.0 / (1.0 + ((x - shifted_center) / width) ** 2),
        "exponential decay": jnp.exp(-2.0 * x),
    }
    imag_curves = {name: _prepare_curve_values(values, positive=positive, zeros=zeros) for name, values in imag_curves.items()}
    return {
        name: jnp.asarray(values, dtype=jnp.complex64) + 1j * jnp.asarray(imag_curves[name], dtype=jnp.complex64)
        for name, values in curves.items()
    }


def _normalise_parameter_names(parameter_names):
    if isinstance(parameter_names, str):
        names = (parameter_names,)
    else:
        names = tuple(parameter_names)

    if not names:
        raise ValueError("At least one parameter name is required.")

    return tuple(str(name) for name in names)


def _normalise_test_variables(test_variables, curve_types=None, complex_values=False, positive=True, zeros=True):
    if isinstance(test_variables, dict):
        specs = []
        for key, config in test_variables.items():
            key = str(key)
            if config is None:
                config = {}
            elif isinstance(config, str):
                config = {"name": config}
            elif not isinstance(config, dict):
                raise TypeError("Variable config must be a dict, string plot name, or None.")

            name = config.get("name", config.get("plot_name", key))
            variable_curve_types = config.get("curve_types", config.get("curve_types_to_test", curve_types))
            variable_complex = config.get("complex_values", config.get("complex", complex_values))
            variable_positive = config.get("positive", positive)
            variable_zeros = config.get("zeros", zeros)

            specs.append(
                TestVariableSpec(
                    key=key,
                    name=str(name),
                    curve_types=variable_curve_types,
                    complex_values=bool(variable_complex),
                    positive=bool(variable_positive),
                    zeros=bool(variable_zeros),
                )
            )

        if not specs:
            raise ValueError("At least one test variable is required.")
        return tuple(specs)

    return tuple(
        TestVariableSpec(
            key=name,
            name=name,
            curve_types=curve_types,
            complex_values=complex_values,
            positive=positive,
            zeros=zeros,
        )
        for name in _normalise_parameter_names(test_variables)
    )


def make_test_dataset(test_variables, length=100, seed=None, curve_types=None, complex_values=False, positive=True, zeros=True):
    variable_specs = _normalise_test_variables(
        test_variables,
        curve_types=curve_types,
        complex_values=complex_values,
        positive=positive,
        zeros=zeros,
    )
    length = int(length)
    if length <= 0:
        raise ValueError("length must be positive.")

    rng = np.random.default_rng(seed)
    parameters = []
    for spec in variable_specs:
        x, available_curves = _curve_templates(length, positive=spec.positive, zeros=spec.zeros)
        if spec.complex_values:
            available_curves = _with_complex_values(available_curves, x, positive=spec.positive, zeros=spec.zeros)

        if spec.curve_types is None:
            selected_curve_types = tuple(available_curves)
        else:
            selected_curve_types = tuple(spec.curve_types)
            unknown = set(selected_curve_types) - set(available_curves)
            if unknown:
                raise ValueError(f"Unknown curve type(s) for {spec.key!r}: {sorted(unknown)}")

        shuffled_curve_types = tuple(rng.permutation(selected_curve_types))
        curves = tuple(
            TestCurve(
                key=spec.key,
                name=spec.name,
                curve_type=curve_type,
                x=x,
                values=available_curves[curve_type],
            )
            for curve_type in shuffled_curve_types
        )
        parameters.append(TestParameter(name=spec.key, plot_name=spec.name, curves=curves))

    return TestDataset(parameters=tuple(parameters))


def make_test_parameter(name, length=100, seed=None, curve_types=None, complex_values=False, positive=True, zeros=True):
    return make_test_dataset(
        name,
        length=length,
        seed=seed,
        curve_types=curve_types,
        complex_values=complex_values,
        positive=positive,
        zeros=zeros,
    )[name]


def _normalise_fixed_args(fixed_args):
    if fixed_args is None:
        return ()
    if isinstance(fixed_args, list):
        return tuple(fixed_args)
    if isinstance(fixed_args, tuple):
        return fixed_args
    return (fixed_args,)


def _params_from_curves(curves):
    return {curve.key: curve.values for curve in curves}


def _prepare_plot_values(values):
    values = np.atleast_1d(np.asarray(values))
    if np.iscomplexobj(values):
        values = np.abs(values)

    x_values = np.linspace(0.0, 1.0, values.shape[0])
    plot_values = values.reshape((values.shape[0], -1))
    if plot_values.shape[1] == 1:
        plot_values = plot_values[:, 0]

    return x_values, plot_values


def _plot_output(ax, output, default_label, label_map):
    if isinstance(output, dict):
        output_items = tuple((str(label_map.get(str(name), name)), values) for name, values in output.items())
        for label, values in output_items:
            x_values, plot_values = _prepare_plot_values(values)
            ax.plot(x_values, plot_values, marker="o", markersize=2, linewidth=1.5, label=label)
        if len(output_items) == 1:
            ax.set_ylabel(output_items[0][0])
        else:
            ax.set_ylabel("")
        if len(output_items) > 1:
            ax.legend(fontsize="small")
        return

    x_values, plot_values = _prepare_plot_values(output)
    ax.plot(x_values, plot_values, marker="o", markersize=2, linewidth=1.5)
    if np.ndim(plot_values) == 1:
        ax.set_ylabel(default_label)
    else:
        ax.set_ylabel("")


def _input_type_title(curves):
    return ", ".join(f"{curve.key}: {curve.curve_type}" for curve in curves)


def _subplot_title(curves, plot_title):
    input_title = _input_type_title(curves)
    if plot_title:
        return f"{plot_title}\n{input_title}"
    return input_title


def _plot_test_results(results, x_axis_name, plot_columns, plot_title, label_map):
    if not results:
        raise ValueError("No test combinations were generated.")

    ncols = min(max(1, int(plot_columns)), len(results))
    nrows = math.ceil(len(results) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), squeeze=False)
    axes = axes.ravel()

    for ax, (curves, output) in zip(axes, results):
        default_label = curves[0].name if len(curves) == 1 else ""
        _plot_output(ax, output, default_label, label_map)
        ax.set_title(_subplot_title(curves, plot_title))
        ax.set_xlabel(x_axis_name)

    for ax in axes[len(results):]:
        ax.set_visible(False)

    fig.tight_layout()
    return fig, axes


def run_random_tests(
    function_under_test,
    test_variables,
    array_length=100,
    random_seed=0,
    curve_types_to_test=None,
    complex_values=False,
    positive=True,
    zeros=True,
    x_axis_name="Time",
    plot_title="",
    plot_columns=3,
    fixed_args_before=(),
    fixed_args_after=(),
    make_plots=True,
):
    fixed_args_before = _normalise_fixed_args(fixed_args_before)
    fixed_args_after = _normalise_fixed_args(fixed_args_after)

    dataset = make_test_dataset(
        test_variables,
        length=array_length,
        seed=random_seed,
        curve_types=curve_types_to_test,
        complex_values=complex_values,
        positive=positive,
        zeros=zeros,
    )
    test_combinations = tuple(zip(*(dataset[name] for name in dataset.names)))
    label_map = {parameter.name: parameter.plot_name for parameter in dataset}

    results = []
    for curves in test_combinations:
        params = _params_from_curves(curves)
        output_array = function_under_test(
            *fixed_args_before,
            params,
            *fixed_args_after
        )
        results.append((curves, output_array))

    results = tuple(results)
    if not results:
        raise ValueError("No test combinations were generated.")

    fig = None
    axes = None
    if make_plots:
        fig, axes = _plot_test_results(
            results,
            x_axis_name=x_axis_name,
            plot_columns=plot_columns,
            plot_title=plot_title,
            label_map=label_map,
        )

    return TestRunResult(dataset=dataset, results=results, fig=fig, axes=axes)


# Example use from another notebook:
# %run ./test.ipynb
# def test_function(params):
#     return <function_to_test>(params["param_1"], params["param_2"])
#
# test_variables = {
#     "param_1": "Parameter 1",
#     "param_2": {"name": "Parameter 2", "positive": False, "complex": True, "zeros": False},
# }
#
# test_run = run_random_tests(
#     test_function,
#     test_variables,
#     array_length=100,
#     plot_title="",
#     plot_columns=3,
#     fixed_args_before=(),
#     fixed_args_after=(),
#     make_plots=True,
# )